In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

df = pd.read_csv('superstore.csv')
conn = sqlite3.connect("superstore.db")

df.to_sql("order", conn, if_exists="replace", index=False)

region_info = pd.DataFrame({
    "Region": ["East", "West", "Central", "South"],
    "Manager": ["Ali", "Sara", "Reza", "Mina"]
})
region_info.to_sql("region_manager", conn, if_exists="replace", index=False)

query1 = """
SELECT
    o.Region,
    o.Category,
    SUM(o.Profit) AS total_profit,
    r.Manager
FROM "order" AS o
JOIN region_manager AS r ON o.Region = r.Region
GROUP BY o.Region, o.Category, r.Manager
"""
#معادل join در sql تابع merge در pandas است
merged=df.merge(region_info,on="Region",how="inner")
result1_pandas=merged.groupby(["Region","Category"]).agg(
    total_profit=("Profit","sum"),Manager=("Manager","first")).reset_index()
result = pd.read_sql_query(query1, conn)
print(result)
print(result1_pandas)
query2 = """
SELECT
    Category,
    Discount,
    CASE
        WHEN Discount >= 0.3 THEN 'high discount'
        WHEN Discount >= 0.1 THEN 'medium discount'
        ELSE 'low discount'
    END AS "discount level",
    Profit
FROM "order"
LIMIT 15
"""
result2 = pd.read_sql_query(query2, conn)
print(result2)

query3="""
SELECT Category,
SUM(Profit) as total_profit
FROM "order"
GROUP BY Category
HAVING SUM(Profit)>1000
"""
#معادل having در pandas استفاده از filter است
grouped=df.groupby(["Category","Discount"])["Profit"].sum().reset_index(name="total_profit")
result3_pandas=grouped[grouped["total_profit"]>1000]

result3=pd.read_sql_query(query3,conn)
print(result3)
print(result3_pandas)
pivot_table_of_df=df.pivot_table(index="Category",columns="Region",values=['Profit', 'Discount'],aggfunc={"Profit":"sum",'Discount': 'mean'})
print(pivot_table_of_df)

     Region         Category  total_profit Manager
0   Central        Furniture     -277.8686    Reza
1   Central  Office Supplies       94.3959    Reza
2   Central       Technology     -900.3958    Reza
3      East        Furniture    -1730.1548     Ali
4      East  Office Supplies     2022.6625     Ali
5      East       Technology      892.7324     Ali
6     South             None           NaN    Mina
7     South        Furniture     -457.3002    Mina
8     South  Office Supplies     -245.3659    Mina
9     South       Technology      213.4902    Mina
10     West        Furniture      634.7740    Sara
11     West  Office Supplies     1411.8218    Sara
12     West       Technology     2938.3528    Sara
     Region         Category  total_profit Manager
0   Central        Furniture     -277.8686    Reza
1   Central  Office Supplies       94.3959    Reza
2   Central       Technology     -900.3958    Reza
3      East        Furniture    -1730.1548     Ali
4      East  Office Supplies   

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=17d9599a-e07d-4c50-b1b1-f0833538d301' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>